**Author**: Felipe Matheus  
**Start Date**: 29/07/2026  
**End Date**: --/07/2026  
**Purpose**: Experiment launcher ("control panel") for the annealing_iacs surrogate.

This notebook does NOT contain pipeline logic. All the logic (Model A -> OOF ->
NNLS -> Model B -> calibration -> metrics -> persistence) lives in
`src/modeling/Experiments.py`, which orchestrates the existing `Modeling` and
`Evaluation` helpers. Here you only:

1. Load and prepare the data (once).
2. Define a base `ExperimentConfig`.
3. Define the grid of variations you want to sweep.
4. Run and inspect the central `experiments_log.csv`.

Results layout on disk:
```
models/annealing_iacs/experiments/
    experiments_log.csv          <- 1 row per run (the "results spreadsheet")
    <tag>__<hash>/               <- 1 folder per run
        config.yaml
        model_a/   model_b/
        artifacts.pkl
        leaderboard_autogluon.csv
```

# 1. Setup

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.metrics.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

c:\Users\fmfoa\Projects\uncertainty-aware-predictors\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TARGET = "elongation_final"
FEATURES = ["initial_diameter", "tensile_strength", "purity", "elongation", "temperature", "time"]
PROCESS = "annealing_elongation"

SCHEMA_DATE = "230726"
FILE_NAME_SCHEMA_DATA = "schema_annealing_essays_{}.csv".format(SCHEMA_DATE)

TAG = f"annealing-schema{SCHEMA_DATE}"
GRID = {
    "time_limit_a": [300, 600],
    # "num_bag_folds_a": [10, 20],
    "features": [
        ("initial_diameter", "tensile_strength", "purity", "elongation", "temperature", "time"),
        ("initial_diameter", "purity", "elongation", "temperature", "time"),
    ],
}

# 2. Data (same preparation as annealing_elongation.ipynb, run once)

In [3]:
df, df_val = proc.process_annealing_elongation(
    features=FEATURES,
    target=TARGET,
    df_schema=pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA))
)

In [ ]:
# # SCHEMA_DATE = "080626"
# # FILE_NAME = "dataset_annealing_iacs.csv"
# FILE_NAME_SCHEMA_DATA = f"schema_annealing_essays_{SCHEMA_DATE}.csv"
# # FILE_NAME_VALIDATION_DATA = "Experimental Results Annealing V2 - CompiledResults.csv"

# # ---- Schema (essay) data: explicit is_essay marker ----
# df_raw_schema = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA))
# df_schema = df_raw_schema[ALL_FEATURES + [TARGET]].dropna()
# df_schema["is_essay"] = True

# # ---- Literature data ----
# df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
# df_float = proc.df_to_float(df_raw, drop_cols=["DOI"], ignore_columns=["material"])
# df_labeled = feng.label_element(df_float).drop_duplicates()
# df_with_masks = feng.add_ratio_mask_column(
#     feng.add_ratio_mask_column(df_labeled, "grain_size"), "iacs",
# )
# df_lit = df_with_masks[df_with_masks.has_Cu == True][ALL_FEATURES + [TARGET]]
# df_lit["is_essay"] = False

# # ---- Validation data ----
# df_val = proc.load_validation_data_chimie_paris(
#     path=os.path.join(varv.PATHS.data_processed, FILE_NAME_VALIDATION_DATA)
# )

# assert df_lit[ALL_FEATURES + [TARGET]].isna().sum().sum() == 0, "NaNs in inputs"

# # ---- Concat. weight_col is built INSIDE the runr from is_essay + config ----
# df = pd.concat([df_schema, df_lit], ignore_index=True)
# print(f"Dataset: {df.shape} | essays: {df.is_essay.sum()} | lit: {(~df.is_essay).sum()}")
# df.head()

Dataset: (96, 6) | essays: 9 | lit: 87


c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(
c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(


,purity,iacs,temperature,time,iacs_final,is_essay
0,99.9,88.260,623.0,30.0,88.350,True
1,99.9,88.260,623.0,60.0,88.655,True
2,99.9,88.260,623.0,90.0,88.800,True
3,99.9,99.825,573.0,30.0,102.340,True
4,99.9,99.390,573.0,30.0,102.230,True


# 3. Base config

In [18]:
df = df[df.iacs > 90]

In [19]:
base = ExperimentConfig(
    process=PROCESS,
    tag=TAG,
    target=TARGET,
    features=tuple(ALL_FEATURES),
    presets_a = "best_quality",
    #     presets_a = "medium_quality"
    #     presets_b = "medium_quality"
    # everything else uses the defaults; override here if needed, e.g.:
    # time_limit_a=120, weight_on_essay_rows=1.0, use_shared_folds=False,
)
print(base.run_id)

annealing-v2-weight-3-best-quality__9e081980


# 4. Single run (sanity check before any grid)

Always run the base config alone first. Then run it 2-3 more times with
`tag="annealing-v1-rep2"` etc. to measure run-to-run noise: AutoGluon under a
time budget is NOT deterministic, and at n~90 this noise is the floor below
which grid differences mean nothing.

In [20]:
result = runr.run_experiment(df, cfg=base, df_val=df_val)
result["artifacts"]["metrics"]

2026-07-07 15:25:12,570 | INFO | src.modeling.Experiments | === Running annealing-v2-weight-3-best-quality__9e081980 ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       12.13 GB / 31.57 GB (38.4%)
Disk Space Avail:   732.26 GB / 932.08 GB (78.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=5, num_bag_sets=1
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report weighted metrics.
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "

{'rmse': 0.8847832475335353,
 'mae': 0.6586034282677284,
 'mape': 0.6571651306970457,
 'r2': 0.8891336496579376}

# 5. Validation set

`df_val` rides along inside the runner: after training + calibration, the
calibrated predictive distribution is evaluated on it and `val_rmse`,
`val_mae`, `val_r2`, `val_cov_*` are appended to the SAME log row — so every
grid run carries fold (OOF) metrics AND validation metrics side by side.

Using `df_schema` as `df_val` is illustrative: those rows are inside the
training data, so val metrics are optimistic. Swap in any held-out
DataFrame with the same columns and nothing else changes.

In [21]:
df_val

,id,purity,initial_diameter,iacs,temperature,time,iacs_final
0,C1_250_30m_T1,99.95,1.2,98.47,523,30,101.08
1,C1_250_30m_T3,99.95,1.2,98.47,523,30,101.08
2,C1_250_60m_T1,99.95,1.2,98.47,523,60,100.82
3,C1_250_60m_T2,99.95,1.2,98.47,523,60,100.82
4,C1_250_90m_T1,99.95,1.2,98.47,523,90,101.22
5,C1_250_90m_T2,99.95,1.2,98.47,523,90,101.22
6,C1_250_30m_AQ_T1,99.95,1.2,98.47,523,30,100.95
7,C1_250_30m_AQ_T2,99.95,1.2,98.47,523,30,100.95
8,C1_300_30m_T1,99.95,1.2,98.47,573,30,102.03
9,C1_300_30m_T2,99.95,1.2,98.47,573,30,102.03


In [22]:
result["artifacts"]["validation_metrics"]

{'rmse': 1.4216301045035884,
 'mae': 1.1201457761156972,
 'mape': 1.1129637440244284,
 'r2': -8.138509538925913,
 'coverage': {0.5: 0.16666666666666666,
  0.8: 0.4444444444444444,
  0.9: 0.5,
  0.95: 0.6666666666666666}}

In [23]:
result["artifacts"].keys()

dict_keys(['config', 'features', 'target', 'base_model_names', 'weights', 'nnls_recovery_ok', 'nnls_max_diff', 'variance_floor', 'recalibration_c', 'ood_ref', 'y_max', 'y_min', 'calibration_before', 'calibration_after', 'aleatoric_diagnostics', 'metrics', 'validation_metrics', 'dataset_hash'])

In [24]:
result.keys()

dict_keys(['cfg', 'run_dir', 'artifacts', 'log_row', 'predictor_a', 'predictor_b'])

In [25]:
result

{'cfg': ExperimentConfig(process='annealing_iacs', tag='annealing-v2-weight-3-best-quality', base_tag=None, target='iacs_final', features=('purity', 'iacs', 'temperature', 'time'), weight_on_essay_rows=1.0, presets_a='best_quality', num_bag_folds_a=5, num_bag_sets_a=1, num_stack_levels_a=0, time_limit_a=120, presets_b='medium_quality', num_bag_folds_b=5, num_stack_levels_b=0, time_limit_b=60, use_weighted_variance=True, variance_floor_frac=0.01, recalibration_target_alpha=0.9, calibration_alphas=(0.5, 0.8, 0.9, 0.95), y_max=None, y_min=None, fold_seed=42, use_shared_folds=False),
 'run_dir': Path('../../models/annealing_iacs/experiments/annealing-v2-weight-3-best-quality/annealing-v2-weight-3-best-quality__9e081980'),
 'artifacts': {'config': {'process': 'annealing_iacs',
   'tag': 'annealing-v2-weight-3-best-quality',
   'base_tag': None,
   'target': 'iacs_final',
   'features': ['purity', 'iacs', 'temperature', 'time'],
   'weight_on_essay_rows': 1.0,
   'presets_a': 'best_quality',

# 6. Grid

Keys are `ExperimentConfig` field names; values are lists of variants.
`features` variants must be tuples. Already-completed runs are skipped
(`force=True` to redo).

In [26]:
log = runr.run_grid(df, base_cfg=base, grid=GRID, df_val=df_val)
log

2026-07-07 15:35:05,350 | INFO | src.modeling.Experiments | Grid: 2 runs over ['time_limit_a', 'num_bag_folds_a', 'weight_on_essay_rows']
2026-07-07 15:35:05,350 | INFO | src.modeling.Experiments | === Running annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=10__weight_on_essay_rows=2.0__a832725d ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       11.48 GB / 31.57 GB (36.4%)
Disk Space Avail:   732.23 GB / 932.08 GB (78.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=10, num_bag_sets=1
Values in column 'weight_col' used as sample weights instead of predictive fea

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,model_dir,cfg_process,cfg_tag,cfg_base_tag,cfg_target,...,mean_sigma_aleat,n_val,val_rmse,val_mae,val_mape,val_r2,val_cov_0.5,val_cov_0.8,val_cov_0.9,val_cov_0.95
0,annealing-v2-weight-3-best-quality__9e081980,2026-07-07T15:35:03,591.1,26,c054a007,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v2-weight-3-best-quality\annealing-v2-weight-3-best-quality__9e081980,annealing_iacs,annealing-v2-weight-3-best-quality,NaN,iacs_final,...,0.27099,36,1.42163,1.12015,1.11296,-8.13851,0.1667,0.4444,0.5000,0.6667
1,annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=10__weight_on_essay_rows=2.0__a832725d,2026-07-07T15:46:19,673.8,26,c054a007,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v2-weight-3-best-quality\annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=10__weight_on_essay_rows=2.0__a832725d,annealing_iacs,annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=10__weight_on_essay_rows=2.0,annealing-v2-weight-3-best-quality,iacs_final,...,0.27099,36,1.31219,0.94355,0.93740,-6.78562,0.4444,0.6111,0.6111,0.7778
2,annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=20__weight_on_essay_rows=2.0__b7afe3bc,2026-07-07T15:57:53,694.1,26,c054a007,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v2-weight-3-best-quality\annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=20__weight_on_essay_rows=2.0__b7afe3bc,annealing_iacs,annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=20__weight_on_essay_rows=2.0,annealing-v2-weight-3-best-quality,iacs_final,...,0.27099,36,1.25733,0.92530,0.91919,-6.14829,0.3333,0.5000,0.5556,0.6111


In [27]:
# import pickle
# from pathlib import Path
# import pandas as pd

# exp_dir = Path("../../models/annealing_iacs/experiments")
# rows = []
# for run_path in exp_dir.iterdir():
#     pkl = run_path / "artifacts.pkl"
#     if not pkl.exists():
#         continue
#     with open(pkl, "rb") as f:
#         art = pickle.load(f)
#     row = {"run_id": run_path.name}
#     row.update({f"cfg_{k}": v for k, v in art["config"].items()})
#     m = dict(art["metrics"]); m.pop("coverage", None)
#     row.update({k: v for k, v in m.items()})
#     cov = art["metrics"].get("coverage", {})
#     row.update({f"cov_{a}": c for a, c in cov.items()})
#     row["c_opt"] = art["recalibration_c"]
#     if art.get("validation_metrics"):
#         vm = dict(art["validation_metrics"]); vcov = vm.pop("coverage", {})
#         row.update({f"val_{k}": v for k, v in vm.items()})
#         row.update({f"val_cov_{a}": c for a, c in vcov.items()})
#     row["cfg_features"] = "|".join(art["features"])
#     rows.append(row)

# pd.DataFrame(rows).to_csv(exp_dir / "experiments_log.csv", index=False)
# print(f"Rebuilt log with {len(rows)} runs")

# 7. Inspect results

In [28]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_weight_on_essay_rows", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9", "c_opt", "pct_truncated_aleat",
    "mean_sigma_epist", "mean_sigma_aleat", "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_weight_on_essay_rows,cfg_features,rmse,mae,cov_0.9,val_rmse,val_mae,val_cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
2,annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=20__weight_on_essay_rows=2.0__b7afe3bc,600,2.0,purity|iacs|temperature|time,0.38467,0.23969,0.8846,1.25733,0.92530,0.5556,1.4549,23.08,0.07389,0.27099,694.1
1,annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=10__weight_on_essay_rows=2.0__a832725d,600,2.0,purity|iacs|temperature|time,0.57513,0.42376,0.9231,1.31219,0.94355,0.6111,1.8139,34.62,0.19646,0.27099,673.8
0,annealing-v2-weight-3-best-quality__9e081980,120,1.0,purity|iacs|temperature|time,0.88478,0.65860,0.8846,1.42163,1.12015,0.5000,2.0122,38.46,0.30484,0.27099,591.1


In [29]:
# Quick pivot: effect of one knob, marginalised over the others.
# Remember: compare against run-to-run noise (Section 4) before concluding.
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse                 mae            cov_0.9          
                     mean       std      mean       std     mean       std
cfg_time_limit_a                                                          
120               0.88478       NaN  0.658600       NaN  0.88460       NaN
600               0.47990  0.134676  0.331725  0.130157  0.90385  0.027224

# 8. Load a winner for deployment / further analysis

Each run folder is self-contained: predictors + artifacts.pkl with weights,
`recalibration_c`, calibration tables.

In [30]:
# log already carries the resolved absolute path, so use it directly
best_row = log.sort_values("rmse").iloc[0]
RUN_ID = best_row["run_id"]
run_dir = Path(best_row["model_dir"])   # <-- absolute path, base_tag included

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=20__weight_on_essay_rows=2.0__b7afe3bc


,alpha,empirical_coverage,gap
0,0.50,0.769231,0.269231
1,0.80,0.807692,0.007692
2,0.90,0.884615,-0.015385
3,0.95,0.961538,0.011538


In [31]:
run_dir

Path('C:/Users/fmfoa/Projects/uncertainty-aware-predictors/models/annealing_iacs/experiments/annealing-v2-weight-3-best-quality/annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=20__weight_on_essay_rows=2.0__b7afe3bc')

In [32]:
run_dir

Path('C:/Users/fmfoa/Projects/uncertainty-aware-predictors/models/annealing_iacs/experiments/annealing-v2-weight-3-best-quality/annealing-v2-weight-3-best-quality__time_limit_a=600__num_bag_folds_a=20__weight_on_essay_rows=2.0__b7afe3bc')